## Breast-Cancer-Diagnostic Dataset

Source: https://www.kaggle.com/competitions/184-702-tu-ml-2025-s-breast-cancer-diagnostic/data

The Breast Cancer Diagnostic Dataset contains 569 tumor samples, each identified by a unique ID. The target variable class indicates whether a tumor is malignant (True) or benign (False). For each sample, a total of 30 numerical measurements are recorded: the arithmetic mean (Mean), the standard error (SE), and the worst (highest) value (Worst) for ten nucleus properties—radius, texture, perimeter, area, smoothness, compactness, concavity, number of concave points, symmetry, and fractal dimension. For example, radiusMean denotes the average radius of the cells in a sample, while radiusWorst gives the maximum measured radius. These statistical features enable analysis of morphological differences and are used in machine-learning models to distinguish between benign and malignant breast cancer tumors.

### Imports

In [ ]:
# Dependencies are pinned in requirements.txt — install with `pip install -r requirements.txt`


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset
import NNutils


### 1. Data Understanding

#### 1.2 Collect Initial Data

In [ ]:
cwd = os.getcwd()

data_folder = os.path.join(cwd, '184-702-tu-ml-2025-s-breast-cancer-diagnostic')

train_path = os.path.join(data_folder, 'breast-cancer-diagnostic.shuf.lrn.csv')

train_df = pd.read_csv(train_path)

print(f"Trainset: {train_df.shape[0]} Rows, {train_df.shape[1]} Columns")
print("Columns in Trainset:", train_df.columns.tolist())
train_df.head()


#### 1.3 Explore Data

In [ ]:
train_df.columns = train_df.columns.str.strip()

##### 1.3.1 Data Description

Data types

In [ ]:
def column_analysis(df : pd.DataFrame):
    dtype_dict = {
        'Column Name': df.columns,
        'Data Type' : df.dtypes,
        'Unique Values' : df.nunique(),
    }
    
    
    dtype_df = pd.DataFrame(dtype_dict).reset_index(drop= True)
    display(dtype_df)    
    
    return dtype_df

analysis_result_df = column_analysis(train_df)

Data Categories


| Variable                   | Description                                                              | Data Category |
|----------------------------|--------------------------------------------------------------------------|---------------|
| ID                         | Unique identifier for each patient/sample                                | Nominal       |
| class                      | Diagnosis label: B = benign, M = malignant                               | Nominal       |
| radiusMean                 | Mean of distances from center to points on the tumor perimeter           | Ratio         |
| textureMean                | Mean of standard deviation of gray‑scale intensity values                | Ratio         |
| perimeterMean              | Mean length of the tumor perimeter                                       | Ratio         |
| areaMean                   | Mean area enclosed by the tumor perimeter                                | Ratio         |
| smoothnessMean             | Mean local variation in radius lengths                                   | Ratio         |
| compactnessMean            | Mean of (perimeter²/area − 1.0), a measure of tumor compactness          | Ratio         |
| concavityMean              | Mean severity of concave portions of the tumor outline                   | Ratio         |
| concavePointsMean          | Mean number of concave points on the tumor contour                       | Ratio         |
| symmetryMean               | Mean symmetry of the tumor                                               | Ratio         |
| fractalDimensionMean       | Mean “coastline approximation” fractal dimension                         | Ratio         |
| radiusStdErr               | Standard error of the radius measurement                                 | Ratio         |
| textureStdErr              | Standard error of the texture measurement                                | Ratio         |
| perimeterStdErr            | Standard error of the perimeter measurement                              | Ratio         |
| areaStdErr                 | Standard error of the area measurement                                   | Ratio         |
| smoothnessStdErr           | Standard error of the smoothness measurement                             | Ratio         |
| compactnessStdErr          | Standard error of the compactness measurement                            | Ratio         |
| concavityStdErr            | Standard error of the concavity measurement                              | Ratio         |
| concavePointsStdErr        | Standard error of the concave points measurement                         | Ratio         |
| symmetryStdErr             | Standard error of the symmetry measurement                              | Ratio         |
| fractalDimensionStdErr     | Standard error of the fractal dimension measurement                      | Ratio         |
| radiusWorst                | “Worst” (mean of three largest values) of radius                         | Ratio         |
| textureWorst               | “Worst” (mean of three largest values) of texture                        | Ratio         |
| perimeterWorst             | “Worst” (mean of three largest values) of perimeter                      | Ratio         |
| areaWorst                  | “Worst” (mean of three largest values) of area                           | Ratio         |
| smoothnessWorst            | “Worst” (mean of three largest values) of smoothness                     | Ratio         |
| compactnessWorst           | “Worst” (mean of three largest values) of compactness                    | Ratio         |
| concavityWorst             | “Worst” (mean of three largest values) of concavity                      | Ratio         |
| concavePointsWorst         | “Worst” (mean of three largest values) of concave points                 | Ratio         |
| symmetryWorst              | “Worst” (mean of three largest values) of symmetry                       | Ratio         |
| fractalDimensionWorst      | “Worst” (mean of three largest values) of fractal dimension              | Ratio         |


Descriptive statistics (numeric)

In [ ]:
display(
    train_df
    .select_dtypes(include=[np.number])
    .agg(['min', 'max', 'mean', 'median', 'std'])
    .T
)

##### 1.3.2 Data Distribution

Distribution

In [ ]:
num_cols = train_df.select_dtypes(include=[np.number]).columns.drop('ID')
fig, axes = plt.subplots(len(num_cols), 2, figsize=(14, 4 * len(num_cols)))
for i, col in enumerate(num_cols):
    sns.histplot(train_df[col], ax=axes[i, 0], kde=True)
    axes[i, 0].set_title(f"{col} – Histogram")
    sns.boxplot(x=train_df[col], ax=axes[i, 1])
    axes[i, 1].set_title(f"{col} – Boxplot")

plt.tight_layout()
plt.show()

Skewness

In [ ]:
print("Skewness per numeric feature:")
for col in num_cols:
    skew_val = skew(train_df[col].dropna())
    interpretation = (
        "(right-skewed)" if skew_val > 0.5 else
        "(left-skewed)" if skew_val < -0.5 else
        "(approximately symmetric)"
    )
    display(f"{col}: {skew_val:.2f} {interpretation}")


##### 1.3.3 Relationships Between Variables

Pearson-Correlation heatmap (no annotation numbers)

In [ ]:
corr = train_df[num_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(
    corr,
    cmap='coolwarm',
    square=True,
    linewidths=0.5,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 6}
)
plt.title("Correlation Matrix of Numeric Features")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

Crosstabs for categorical features (excluding 'class')

-> There are no categorical variables except 'class'

##### 1.3.4 Identify Missing Values

In [ ]:
missing = train_df.isna().sum()
print(missing)
print("Absolute missing values:")
print(missing[missing > 0])
print("\nPercentage missing values:")
print((missing / len(train_df) * 100).round(2)[missing > 0])

##### 1.3.5 Data Quality Checks

Duplicate rows

In [ ]:
dups = train_df.duplicated().sum()
print(f"Duplicate rows: {dups}")

Plausibility check (negative values)

In [ ]:
for col in num_cols:
    if (train_df[col] < 0).any():
        print(f"Negative values found in '{col}'")
    else:
        print('No negative values')

##### 1.3.6 Additional Visualizations

In [ ]:
#sns.countplot(x='class', data=train_df)
#plt.title("Distribution of Target Variable 'class'")
#plt.show()

### 2. Data Preperation

#### 2.1 Select Data

In [ ]:
X = train_df.drop(columns=['ID', 'class'])
y = train_df['class']

# Hold out test set (used ONCE at the end on the best grid config)
X_trainval_raw, X_test_raw, y_trainval_raw, y_test_raw = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Carve a validation split out of the remaining train data — grid search uses this
X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    X_trainval_raw, y_trainval_raw,
    test_size=0.2,
    stratify=y_trainval_raw,
    random_state=42
)


#### 2.2 Clean Data

##### 2.2.1 Handle Duplicates

No duplicate rows in train_df

##### 2.2.2 Handle Missing Values

No missing values in any column

##### 2.2.3 Handle Outliers

To limit the undue influence of isolated extreme values in the dataset, we apply Winsorization (clipping) at the 1st and 99th percentiles of each numeric variable. Values below the 1st percentile are raised to that threshold, and values above the 99th percentile are lowered to the corresponding threshold. Unlike removing outliers, this procedure retains all observations—leaving the sample size unchanged—while reducing variance inflation caused by extreme values and preserving the overall distribution shape. For example, the attribute *fractalDimensionWorst* exhibits outliers at approximately 0.175 and 0.22 in both the histogram and the boxplot (see Section 1.3.2). Since these values still fall within a plausible, realistic range, it is more appropriate to clip them rather than remove them entirely.

In [ ]:
def cap_outliers(df, lower=0.01, upper=0.99):
    df2 = df.copy()
    for col in df2.columns:
        lo, hi = df2[col].quantile([lower, upper])
        df2[col] = df2[col].clip(lo, hi)
    return df2

X_clean = cap_outliers(X_train_raw)

##### 2.2.4 Handle Skewed Distributions

As part of the data exploration, the distribution of all numeric features was examined (see chapter 1.3.2 Data Distributionn). It was observed that almost all variables exhibit a right‑skewed distribution, with the exception of **textureWorst**, whose distribution is approximately symmetric. Some attributes, such as **symmetryStdErr** (skewness: 2.15) and **fractalDimensionWorst** (skewness: 1.99), display particularly pronounced skewness. Despite these observations, no data transformation is applied at this stage, as the subsequent analysis will first assess how various classification models perform on the original distributions.

#### 2.3 Construct Data

Since the existing features are already meaningful aggregated metrics (Mean, Standard Error, “Worst” values) that capture various aspects of tumor structure, no additional feature construction is necessary for this dataset.

#### 2.4 Integrate Data

Since all relevant information is contained within the provided training and test files and there are no additional external sources to merge, the Data Integration step is not required for this dataset.

#### 2.5 Format Data

In [ ]:
X_clean.columns     = X_clean.columns.str.strip()
X_val_raw.columns   = X_val_raw.columns.str.strip()
X_test_raw.columns  = X_test_raw.columns.str.strip()


In [ ]:
list_ratio = [
    'radiusMean', 'textureMean', 'perimeterMean', 'areaMean', 'smoothnessMean',
    'compactnessMean', 'concavityMean', 'concavePointsMean', 'symmetryMean', 'fractalDimensionMean',
    'radiusStdErr', 'textureStdErr', 'perimeterStdErr', 'areaStdErr', 'smoothnessStdErr',
    'compactnessStdErr', 'concavityStdErr', 'concavePointsStdErr', 'symmetryStdErr', 'fractalDimensionStdErr',
    'radiusWorst', 'textureWorst', 'perimeterWorst', 'areaWorst', 'smoothnessWorst',
    'compactnessWorst', 'concavityWorst', 'concavePointsWorst', 'symmetryWorst', 'fractalDimensionWorst'
]

list_nominal = ['class']

In [ ]:
X_train = X_clean[list_ratio].copy()
X_val   = X_val_raw[list_ratio].copy()
X_test  = X_test_raw[list_ratio].copy()


##### 2.5.1 Encode Attibutes

In [ ]:
le = LabelEncoder()
y_train_enc = pd.Series(
    le.fit_transform(y_train_raw),
    name='class', index=y_train_raw.index
)
y_val_enc = pd.Series(
    le.transform(y_val_raw),
    name='class', index=y_val_raw.index
)
y_test_enc = pd.Series(
    le.transform(y_test_raw),
    name='class', index=y_test_raw.index
)


##### 2.5.2 Scale Data

StandardScaler (Z-scaler)

In [ ]:
std_scaler = StandardScaler()
X_train_std = pd.DataFrame(
    std_scaler.fit_transform(X_train),
    columns=list_ratio, index=X_train.index
)
X_val_std = pd.DataFrame(
    std_scaler.transform(X_val),
    columns=list_ratio, index=X_val.index
)
X_test_std = pd.DataFrame(
    std_scaler.transform(X_test),
    columns=list_ratio, index=X_test.index
)


In [ ]:
#X_train_std.describe()

In [ ]:
#X_test_std.describe()

### 3. Datasets

#### 3.1 Raw Datasets

In [ ]:
#display(X_train.head())
#display(X_test.head())

#### 3.2 Prepared Datasets

In [ ]:
print("=== Dataset StandardScaler ===")
display(X_train_std.head())
display(X_test_std.head())

print("=== Dataset Target_Variable ===")
display(y_train_enc.head())
display(y_test_enc.head())

### 4. Modeling

#### 4.1 Select Modeling Technique

##### 4.1.1 Modeling Technique

##### 4.1.2 Metrics

#### 4.2 Generate Test Design

#### 4.3 Build Model

In [ ]:
from sklearn.preprocessing import OneHotEncoder

X_train_np = X_train_std.values
X_val_np   = X_val_std.values
X_test_np  = X_test_std.values

ohe = OneHotEncoder(sparse_output=False)
y_train_ohe = ohe.fit_transform(y_train_enc.values.reshape(-1, 1))
y_val_ohe   = ohe.transform(y_val_enc.values.reshape(-1, 1))
y_test_ohe  = ohe.transform(y_test_enc.values.reshape(-1, 1))


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

def plot_loss_and_cm(cm, loss_history):
    labels = ["Benign", "Malignant"]
    
    plt.figure(figsize=(12, 5))
    
    # Loss curve subplot
    plt.subplot(1, 2, 1)
    plt.plot(loss_history)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training Loss")
    
    # Confusion matrix subplot
    plt.subplot(1, 2, 2)
    
    max_val = 50
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='rocket', xticklabels=labels, yticklabels=labels,vmin=0, vmax=max_val)
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.title(f"Confusion Matrix")
    
    plt.tight_layout()
    plt.show()
        

## Experiment Helper functions

##### 4.3.1 Scratch_NN

In [ ]:
from Custom_nn import NeuralNetwork

# Class weights for imbalanced classes
counts   = np.bincount(y_train_enc.values)
inv_freq = 1.0 / counts
norm     = inv_freq * (len(counts) / inv_freq.sum())
cw_dict  = {label: norm[label] for label in range(len(counts))}

# Transpose so X has shape (n_x, m) and Y has shape (n_y, m)
X_train_mat = X_train_np.T
Y_train_mat = y_train_ohe.T


def fit_custom_nn(model, X_eval, y_eval_labels):
    """Train on (X_train_mat, Y_train_mat); evaluate on the supplied eval set."""
    loss_history = []
    start = time.perf_counter()

    for epoch in range(1, model.epochs + 1):
        A_L, cache = model.feed_forward(X_train_mat)
        loss = model.compute_cost(A_L, Y_train_mat)
        loss_history.append(loss)
        grads_W, grads_b = model.backprop(cache, Y_train_mat)
        model.update_parameters(grads_W, grads_b)

    training_time = time.perf_counter() - start

    # Evaluate on supplied set — argmax over softmax-like sigmoid outputs
    A_eval, _     = model.feed_forward(X_eval.T)
    y_pred_labels = np.argmax(A_eval, axis=0)

    cm        = confusion_matrix(y_eval_labels, y_pred_labels)
    accuracy  = accuracy_score(y_eval_labels, y_pred_labels)
    precision = precision_score(y_eval_labels, y_pred_labels)
    recall    = recall_score(y_eval_labels, y_pred_labels)

    return accuracy, precision, recall, loss_history, cm, training_time


#### Keynotes:

uniform activation function for each grid aka. every hidden layer uses the same activation function (except output layer which uses sigmoid)

uniform neurons per layer. all hidden layers have the same number of hidden layers, otherwise amount of combinations would be too large 

In [ ]:
from NNutils import count_learnable_params, estimate_vram


def grid_search_custom_nn():
    """
    Grid search over depth, width, activation, lr, epochs.
    Selection metric: validation accuracy (test set held out).
    """
    import warnings
    warnings.filterwarnings('ignore', category=UserWarning)
    warnings.filterwarnings('ignore', category=RuntimeWarning)

    num_layers_grid = [1, 2, 3, 4]
    width_presets = {
        1: [(16,), (32,)],
        2: [(16, 16), (8, 16), (16, 8)],
        3: [(16, 16, 16), (8, 16, 32), (32, 16, 8), (16, 8, 16)],
        4: [(16, 16, 16, 16), (32, 16, 16, 8), (8, 8, 16, 32), (16, 32, 32, 16)],
    }
    activation_grid = ['relu', 'sigmoid', 'tanh']
    lr_grid = [0.01, 0.05]
    epochs_grid = [100, 200, 500]

    results = []

    for num_layers in num_layers_grid:
        for neurons_combo in width_presets[num_layers]:
            layer_sizes = [X_train_np.shape[1]] + list(neurons_combo) + [2]
            for activation in activation_grid:
                for lr in lr_grid:
                    for epochs in epochs_grid:
                        activations = [activation] * num_layers + ['sigmoid']

                        model = NeuralNetwork(
                            layer_sizes=layer_sizes,
                            activations=activations,
                            lr=lr,
                            epochs=epochs,
                            class_weights=cw_dict,
                            seed=42,
                        )

                        params = count_learnable_params(model)
                        vram   = estimate_vram(params)

                        # Evaluate on VAL set during grid search (test is held out)
                        accuracy, precision, recall, loss_history, cm, train_time = fit_custom_nn(
                            model, X_val_np, y_val_enc.values
                        )

                        results.append({
                            'num_layers': num_layers,
                            'neurons_pattern': neurons_combo,
                            'activation': activation,
                            'lr': lr,
                            'epochs': epochs,
                            'val_accuracy': accuracy,
                            'val_precision': precision,
                            'val_recall': recall,
                            'vram_mb': vram,
                            'params': params,
                            'fit_time': train_time,
                            'time_per_epoch': train_time / epochs,
                            'loss_history': loss_history,
                            'confusion_matrix': cm,
                        })

    results_df = pd.DataFrame(results)

    best_idx  = results_df['val_accuracy'].idxmax()
    worst_idx = results_df['val_accuracy'].idxmin()
    best_config  = results_df.iloc[best_idx]
    worst_config = results_df.iloc[worst_idx]

    return results_df, best_config, worst_config


#### Initial Performance analysis Best & worst 

In [ ]:
results_df, best_config, worst_config = grid_search_custom_nn()

print(f"{'='*100}\nGrid-Search Results (validation set)\n{'='*100}")
display(results_df)

cm_best   = best_config['confusion_matrix']
loss_best = best_config['loss_history']
cm_worst  = worst_config['confusion_matrix']
loss_worst = worst_config['loss_history']

# Final test-set evaluation: refit best config and report TEST metrics ONCE
best_layer_sizes = [X_train_np.shape[1]] + list(best_config['neurons_pattern']) + [2]
best_activations = [best_config['activation']] * best_config['num_layers'] + ['sigmoid']
final_model = NeuralNetwork(
    layer_sizes=best_layer_sizes,
    activations=best_activations,
    lr=best_config['lr'],
    epochs=int(best_config['epochs']),
    class_weights=cw_dict,
    seed=42,
)
test_acc, test_prec, test_rec, _, test_cm, _ = fit_custom_nn(
    final_model, X_test_np, y_test_enc.values
)
print(f"\n{'='*100}\nFinal held-out TEST metrics for best config\n{'='*100}")
print(f"Accuracy : {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall   : {test_rec:.4f}")


In [ ]:
print(f"{'='*100}\nAnalysis Best Model\n{'='*100}")
plot_loss_and_cm(cm_best, loss_best)
display(best_config)
print(f"{'='*100}\nWorst Model\n{'='*100}")
plot_loss_and_cm(cm_worst, loss_worst)
display(worst_config)

### CustomNN Grid Analysis

In [ ]:
"""
One‑shot script that draws every diagnostic figure *immediately* – no helpers
needed.  Run this cell **after** executing `grid_search_custom_nn()` so that
`results_df`, `best_config`, `worst_config`, and the training data variables are
in scope.

Figures drawn
-------------
1. Scatter – validation accuracy vs number of hidden layers (hue = activation)
2. Box‑plot – accuracy grouped by learning rate
3. Box‑plot – accuracy grouped by epoch cap
4. Scatter – VRAM usage vs **parameter count** (colour = depth)
5. Training‑loss curves – best run *per activation function*
6. Confusion matrices – best vs worst model (raw counts only)
"""

sns.set_theme(style="whitegrid")

# ------------------------------------------------------------------
#  Utility: compute parameter count if not already present
# ------------------------------------------------------------------
if "param_count" not in results_df.columns:
    input_dim = X_train_np.shape[1]
    def _param_count(row):
        sizes = [input_dim] + list(row["neurons_pattern"]) + [2]
        cnt = 0
        for i in range(len(sizes) - 1):
            cnt += sizes[i] * sizes[i + 1] + sizes[i + 1]
        return cnt
    results_df["param_count"] = results_df.apply(_param_count, axis=1)

# ------------------------------------------------------------------
# 1. Scatter – accuracy vs depth (colour = activation)
# ------------------------------------------------------------------
plt.figure(figsize=(6, 4))
sns.scatterplot(data=results_df,
                x="num_layers", y="val_accuracy",
                hue="activation", s=60)
plt.title("Validation accuracy vs hidden‑layer depth")
plt.xlabel("Number of hidden layers")
plt.ylabel("Accuracy")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 2. Box‑plot – accuracy by learning rate
# ------------------------------------------------------------------
plt.figure(figsize=(6, 4))
sns.boxplot(data=results_df, x="lr", y="val_accuracy", color="lightgray")
plt.title("Accuracy distribution by learning rate")
plt.xlabel("Learning rate")
plt.ylabel("Accuracy")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 3. Box‑plot – accuracy by epoch cap
# ------------------------------------------------------------------
plt.figure(figsize=(6, 4))
sns.boxplot(data=results_df, x="epochs", y="val_accuracy", color="lightgray")
plt.title("Accuracy distribution by epoch cap")
plt.xlabel("Epoch cap")
plt.ylabel("Accuracy")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 4. Scatter – VRAM usage vs parameter count (colour = depth)
# ------------------------------------------------------------------
plt.figure(figsize=(6, 4))
sns.scatterplot(data=results_df, x="param_count", y="vram_mb",
                hue="num_layers", palette="viridis", s=60)
plt.title("VRAM usage vs model size (parameter count)")
plt.xlabel("Parameter count")
plt.ylabel("Estimated VRAM (MB)")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 5. Loss curves – best run per activation
# ------------------------------------------------------------------
plt.figure(figsize=(6, 4))
for act in results_df["activation"].unique():
    subset = results_df[results_df["activation"] == act]
    best_row = subset.loc[subset["val_accuracy"].idxmax()]
    loss_curve = best_row["loss_history"]
    plt.plot(range(1, len(loss_curve) + 1), loss_curve,
             label=f"{act} acc={best_row['val_accuracy']:.3f}")
plt.title("Training loss – best run per activation function")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 6. Confusion matrices – best vs worst (raw counts only)
# ------------------------------------------------------------------
labels = ["Negative", "Positive"]

best_row = results_df.loc[results_df["val_accuracy"].idxmax()]
worst_row = results_df.loc[results_df["val_accuracy"].idxmin()]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, row, ttl in zip(axes, [best_row, worst_row], ["Best", "Worst"]):
    cm = row["confusion_matrix"]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                cbar=False, xticklabels=labels, yticklabels=labels)
    ax.set_title(f"{ttl} model – raw counts")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.tight_layout()
plt.show()

## Todo

1. Evaluation function
- Plot: loss curve, confusion matrix, precision-recall curve



### 4.3.2 LLM_NN Implementation

In [ ]:
from LLM_nn import LLMNeuralNetwork

X_train_np = X_train_std.values
X_test_np  = X_test_std.values
y_train_labels = y_train_enc.values
y_test_labels  = y_test_enc.values

# Class weights
counts   = np.bincount(y_train_labels)
inv_freq = 1.0 / counts
norm     = inv_freq * (len(counts) / inv_freq.sum())
class_weights = {i: norm[i] for i in range(len(counts))}

n_x = X_train_np.shape[1]
n_y = len(np.unique(y_train_labels))
dims = [n_x, 16, 16, n_y]

model_llm = LLMNeuralNetwork(
    dims=dims,
    activation='relu',
    lr=0.01,
    class_weights=class_weights,
    seed=42,
)


def NN_LLM(model_llm):
    loss_history = []
    epochs = 500
    start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        AL, cache = model_llm.forward(X_train_np)
        loss = model_llm.compute_loss(AL, y_train_labels)
        loss_history.append(loss)
        grads = model_llm.backward(X_train_np, y_train_labels, cache)
        model_llm.update(grads)

    training_time = time.perf_counter() - start

    y_pred = model_llm.predict(X_test_np)
    accuracy  = accuracy_score(y_test_labels, y_pred)
    precision = precision_score(y_test_labels, y_pred)
    recall    = recall_score(y_test_labels, y_pred)
    cm        = confusion_matrix(y_test_labels, y_pred)

    params = count_learnable_params(model_llm)
    vram   = estimate_vram(params)

    results_dict = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'vram_mb': vram,
        'parameters': params,
        'fit_time': training_time,
        'time_per_epoch': training_time / epochs,
        'confusion_matrix': cm,
        'loss_history': loss_history,
    }

    return pd.DataFrame([results_dict])


### LLM analysis

In [ ]:
LLM_result_df = NN_LLM(model_llm)

display(LLM_result_df)
plot_loss_and_cm(LLM_result_df['confusion_matrix'].iloc[0], LLM_result_df['loss_history'].iloc[0])

### 4.3.3 PyTorch_NN Implementation

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, precision_score, recall_score

import torch

from PyTorch_nn import TorchModel


X_train_np = X_train_std.values
X_test_np  = X_test_std.values
y_train_labels = y_train_enc.values  # Shape (m_train,)
y_test_labels  = y_test_enc.values   # Shape (m_test,)

# calculate weights
counts   = np.bincount(y_train_labels)
inv_freq = 1.0 / counts
norm     = inv_freq * (len(counts) / inv_freq.sum())
cw_dict  = {i: norm[i] for i in range(len(counts))}


# Convert to tensors
X_train_t = torch.FloatTensor(X_train_np)
y_train_t = torch.LongTensor(y_train_labels)
X_test_t = torch.FloatTensor(X_test_np)
y_test_t = torch.LongTensor(y_test_labels)

# DataLoaders
train_ds = TensorDataset(X_train_t, y_train_t)
test_ds  = TensorDataset(X_test_t, y_test_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=32)

# Model, loss, optimizer
model_torch = TorchModel(
    input_features = X_train_np.shape[1],
    h1             = 16,
    h2             = 16,
    output_features= len(np.unique(y_train_labels)),
    class_weights  = cw_dict
)

def torch_model_fit(model_torch):

    optimizer = torch.optim.SGD(model_torch.parameters(), lr=0.01)
    
    # Training loop
    loss_history = []
    start_time = time.time()
    epochs = 100
    
    for epoch in range(1, epochs+1):
        model_torch.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            outputs = model_torch(inputs)
            loss    = model_torch.compute_loss(outputs, labels)
            model_torch.backward(loss, optimizer)
            running_loss += loss.item() * inputs.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        loss_history.append(epoch_loss)
        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d} — Loss: {epoch_loss:.6f}")
    
    training_time = time.time() - start_time
    print(f"\nPyTorch NN Training duration: {training_time:.1f} s")
    
    # loss curve
    plt.plot(loss_history)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("PyTorch NN Training Loss")
    plt.show()
    
    # Evaluation
    model_torch.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model_torch(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())
    
    accuracy  = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall    = recall_score(all_labels, all_preds)
    


### PyTorch analysis

## Experiment Comparisson 